# Machine Translation (German → English) - Seq2Seq with GRU


In [ ]:
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
import torchtext.datasets as datasets
from typing import Iterable, List
from torch.nn.utils.rnn import pad_sequence
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader

print(f"PyTorch version: {torch.__version__}")

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

In [ ]:
# Language pair
SRC_LANGUAGE = 'de'  # German
TGT_LANGUAGE = 'en'  # English

In [ ]:
# Install spacy models if not already installed
# Run these commands in terminal:
# pip install spacy
# python -m spacy download en_core_web_sm
# python -m spacy download de_core_news_sm

In [ ]:
# Setup tokenizers
de_tokenizer = get_tokenizer('spacy', language='de_core_news_sm')
en_tokenizer = get_tokenizer('spacy', language='en_core_web_sm')

print("Tokenizers initialized")

In [ ]:
# Function to yield tokens for vocabulary building
def yield_tokens(data_iter: Iterable, language: str) -> List[str]:
    for data_sample in data_iter:
        if language == 'de':
            yield de_tokenizer(data_sample[0])
        elif language == 'en':
            yield en_tokenizer(data_sample[1])

# Define special symbols and indices
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ['<unk>', '<pad>', '<bos>', '<eos>']

In [ ]:
# Load and build German vocabulary
print("Loading Multi30k dataset and building vocabularies...")
train_iter = datasets.Multi30k(split='train', language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))

vocab_de = build_vocab_from_iterator(
    yield_tokens(train_iter, 'de'),
    min_freq=1,
    specials=special_symbols,
    special_first=True
)
vocab_de.set_default_index(UNK_IDX)

print(f"German vocabulary size: {len(vocab_de)}")

In [ ]:
# Build English vocabulary
train_iter = datasets.Multi30k(split='train', language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))

vocab_en = build_vocab_from_iterator(
    yield_tokens(train_iter, 'en'),
    min_freq=1,
    specials=special_symbols,
    special_first=True
)
vocab_en.set_default_index(UNK_IDX)

print(f"English vocabulary size: {len(vocab_en)}")

In [ ]:
# Collate function for DataLoader
def collate_fn(batch):
    src_batch, tgt_batch = [], []
    
    for src_sample, tgt_sample in batch:
        src_sample = src_sample.rstrip("\n")
        tgt_sample = tgt_sample.rstrip("\n")
        
        src_tokens = de_tokenizer(src_sample)
        tgt_tokens = en_tokenizer(tgt_sample)
        
        src_ids = vocab_de(src_tokens)
        tgt_ids = vocab_en(tgt_tokens)
        
        src_ids.append(EOS_IDX)
        tgt_ids.append(EOS_IDX)
        tgt_ids.insert(0, BOS_IDX)
        
        src_tensor = torch.tensor(src_ids)
        tgt_tensor = torch.tensor(tgt_ids)
        
        src_batch.append(src_tensor)
        tgt_batch.append(tgt_tensor)
    
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX, batch_first=True)
    
    return src_batch, tgt_batch

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_size, embed_size, hidden_size, dropout_p=0.1):
        super().__init__()
        self.e = nn.Embedding(input_size, embed_size)
        self.dropout = nn.Dropout(dropout_p)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)
    
    def forward(self, x):
        x = self.e(x)
        x = self.dropout(x)
        outputs, hidden = self.gru(x)
        return outputs, hidden

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_size, embed_size, hidden_size):
        super().__init__()
        self.e = nn.Embedding(output_size, embed_size)
        self.relu = nn.ReLU()
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)
        self.lin = nn.Linear(hidden_size, output_size)
        self.lsoftmax = nn.LogSoftmax(dim=-1)
    
    def forward(self, x, prev_hidden):
        x = self.e(x)
        x = self.relu(x)
        output, hidden = self.gru(x, prev_hidden)
        y = self.lin(output)
        y = self.lsoftmax(y)
        return y, hidden

In [ ]:
def train_one_epoch(encoder, decoder, train_dataloader, opte, optd, loss_fn, device):
    encoder.train()
    decoder.train()
    track_loss = 0
    
    for i, (s_ids, t_ids) in enumerate(train_dataloader):
        s_ids = s_ids.to(device)
        t_ids = t_ids.to(device)
        
        encoder_outputs, encoder_hidden = encoder(s_ids)
        decoder_hidden = encoder_hidden
        yhats, decoder_hidden = decoder(t_ids[:, 0:-1], decoder_hidden)
        
        gt = t_ids[:, 1:]
        yhats_reshaped = yhats.view(-1, yhats.shape[-1])
        gt = gt.reshape(-1)
        
        loss = loss_fn(yhats_reshaped, gt)
        track_loss += loss.item()
        
        opte.zero_grad()
        optd.zero_grad()
        loss.backward()
        opte.step()
        optd.step()
    
    return track_loss / (i + 1)

In [ ]:
def eval_one_epoch(encoder, decoder, val_dataloader, loss_fn, device, vocab_de, vocab_en, e, n_epochs):
    encoder.eval()
    decoder.eval()
    track_loss = 0
    
    with torch.no_grad():
        for i, (s_ids, t_ids) in enumerate(val_dataloader):
            s_ids = s_ids.to(device)
            t_ids = t_ids.to(device)
            
            encoder_outputs, encoder_hidden = encoder(s_ids)
            decoder_hidden = encoder_hidden
            input_id = t_ids[:, 0]
            yhats = []
            
            if e + 1 == n_epochs:
                pred_sentence = ""
            
            for j in range(1, t_ids.shape[1]):
                probs, decoder_hidden = decoder(input_id.unsqueeze(1), decoder_hidden)
                yhats.append(probs)
                _, input_id = torch.topk(probs, 1, dim=-1)
                input_id = input_id.squeeze(1, 2)
                
                if e + 1 == n_epochs:
                    word = vocab_en.lookup_token(input_id.item())
                    pred_sentence += word + " "
                
                if input_id.item() == EOS_IDX:
                    break
            
            if e + 1 == n_epochs and i < 5:  # Print first 5 examples
                src_sentence_tokens = vocab_de.lookup_tokens(s_ids.tolist()[0])
                src_sentence = " ".join(src_sentence_tokens)
                gt_sentence_tokens = vocab_en.lookup_tokens(t_ids[:, 1:].tolist()[0])
                gt_sentence = " ".join(gt_sentence_tokens)
                print("\n" + "-" * 50)
                print(f"Source: {src_sentence}")
                print(f"Target: {gt_sentence}")
                print(f"Predicted: {pred_sentence}")
            
            yhats_cat = torch.cat(yhats, dim=1)
            yhats_reshaped = yhats_cat.view(-1, yhats_cat.shape[-1])
            gt = t_ids[:, 1:j+1]
            gt = gt.view(-1)
            
            loss = loss_fn(yhats_reshaped, gt)
            track_loss += loss.item()
    
    if e + 1 == n_epochs:
        print("-" * 50)
    
    return track_loss / (i + 1)

In [ ]:
# Hyperparameters
embed_size = 300
hidden_size = 512
batch_size = 32
n_epochs = 10  # Reduced from 20 for faster training
lr = 0.001

# Initialize models
encoder = Encoder(len(vocab_de), embed_size, hidden_size).to(device)
decoder = Decoder(len(vocab_en), embed_size, hidden_size).to(device)

# Loss and optimizers
loss_fn = nn.NLLLoss(ignore_index=PAD_IDX).to(device)
opte = optim.Adam(params=encoder.parameters(), lr=lr, weight_decay=0.001)
optd = optim.Adam(params=decoder.parameters(), lr=lr, weight_decay=0.001)

print(f"Models initialized on {device}")
print(f"German vocab: {len(vocab_de)}, English vocab: {len(vocab_en)}")

In [ ]:
# Training loop
print("\nStarting training...\n")

for e in range(n_epochs):
    # Create fresh iterators for each epoch
    train_iter = datasets.Multi30k(split='train', language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
    train_dataloader = DataLoader(train_iter, batch_size=batch_size, collate_fn=collate_fn)
    
    val_iter = datasets.Multi30k(split='valid', language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
    val_dataloader = DataLoader(val_iter, batch_size=1, collate_fn=collate_fn)
    
    print(f"Epoch {e+1}/{n_epochs}", end=", ")
    
    train_loss = train_one_epoch(encoder, decoder, train_dataloader, opte, optd, loss_fn, device)
    print(f"Train Loss: {train_loss:.4f}", end=", ")
    
    eval_loss = eval_one_epoch(encoder, decoder, val_dataloader, loss_fn, device, vocab_de, vocab_en, e, n_epochs)
    print(f"Val Loss: {eval_loss:.4f}")

## Notes

**Improvements possible:**
- Add attention mechanism (see pract7 and pract8)
- Better architecture (bidirectional encoder, deeper networks)
- Better training (learning rate scheduling, gradient clipping)
- Address overfitting (dropout, regularization)
- Beam search for decoding